<a href="https://colab.research.google.com/github/GabrielJ07/ConfiguratorAgent/blob/main/Convert_MyActivity_to_Markdown.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import re
import html
from datetime import datetime
import os

def clean_html(raw_html):
    """Removes HTML tags and decodes HTML entities for clean text output."""
    if not raw_html:
        return ""

    # Unescape HTML entities (e.g., &#39; to ')
    text = html.unescape(raw_html)

    # Replace <br> and <p> with newlines for readability
    text = re.sub(r'<br\s*/?>', '\n', text, flags=re.IGNORECASE)
    text = re.sub(r'</p>\s*<p>', '\n\n', text, flags=re.IGNORECASE)

    # Remove all remaining HTML tags
    cleanr = re.compile('<.*?>')
    cleantext = re.sub(cleanr, '', text)

    return cleantext.strip()

def convert_myactivity_to_md(input_file, output_file):
    """Parses Google Takeout MyActivity JSON and converts it to NotebookLM-friendly Markdown."""

    if not os.path.exists(input_file):
        print(f"Error: Could not find '{input_file}'. Make sure it is in the same folder as this script.")
        return

    print(f"Reading {input_file}...")
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    markdown_lines = ["# Gemini Activity History\n\n"]
    entry_count = 0

    for item in data:
        # We only want to process actual conversations, skipping generic app opens
        title = item.get('title', '')
        if not title:
            continue

        # Clean up the "Prompted " or "Said " prefix Google Takeout automatically adds
        if title.startswith("Prompted "):
            title = title[9:]
        elif title.startswith("Said "):
            title = title[5:]

        # Format the timestamp
        time_str = item.get('time', '')
        try:
            # Convert ISO time to a readable format
            dt = datetime.fromisoformat(time_str.replace('Z', '+00:00'))
            formatted_time = dt.strftime("%B %d, %Y at %I:%M %p")
        except ValueError:
            formatted_time = time_str

        # Extract and clean the AI's response
        safe_html_items = item.get('safeHtmlItem', [])
        response_html = "".join([f"{html_item['html']}\n" for html_item in safe_html_items if 'html' in html_item])

        response_text = clean_html(response_html)

        # Only add entries that have actual text
        if title or response_text:
            markdown_lines.append(f"## Date: {formatted_time}\n")
            markdown_lines.append(f"**User Prompt:**\n{title}\n\n")
            if response_text:
                markdown_lines.append(f"**Gemini Response:**\n{response_text}\n\n")
            markdown_lines.append("---\n\n")
            entry_count += 1

    print(f"Writing to {output_file}...")
    with open(output_file, 'w', encoding='utf-8') as f:
        f.writelines(markdown_lines)

    print(f"Success! Converted {entry_count} conversation entries.")
    print(f"You can now upload '{output_file}' directly into NotebookLM.")

if __name__ == "__main__":
    # Ensure these file names match your local files
    INPUT_FILENAME = 'MyActivity.json'
    OUTPUT_FILENAME = 'Circuittelligence_Activity.md'

    convert_myactivity_to_md(INPUT_FILENAME, OUTPUT_FILENAME)